# NatureCubePy Data Upload Tutorial

This notebook demonstrates safe, practical upload workflows for NatureCubePy.

The examples default to dry-run mode so you can inspect payloads before sending any writes to the API.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import uuid

import pandas as pd

from naturecubepy import (
    add_project_labels,
    auth_headers,
    build_device_settings,
    build_feature_record,
    build_observation,
    check_edna_labels_df,
    get_camera_trap_data,
    get_key,
    get_procedure,
    get_project,
    get_project_labels,
    get_project_systems,
    get_station_info,
    get_media_assets_df,
    list_systems,
    split_multi_value_choices,
    set_segment_published_status,
    update_media_timestamps,
    upload_edna_records,
    upload_observations_from_csv,
    upload_phone_observations,
    validate_csv_against_procedure,
    validate_observation_payload,
)

# Keep False until you are ready to write to the API.
RUN_UPLOADS = False
TUTORIAL_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path("tutorials")


In [2]:
# Retrieve API key and set up authentication headers
api_key = get_key('EV_NI_PROD')
hdr = auth_headers(api_key, okala_url='https://naturecube.io/api')
project_name = get_project(hdr)

Retrieving project data...
Received response with status code 200
Project data retrieved successfully
Setting your active project as - NI TEST


## 1. Upload Project Labels

This example creates `Label` payloads and uploads them with `add_project_labels(...)`.

To keep the example realistic and safe, we reuse a few existing camera labels from your project.

In [ ]:
camera_labels = get_project_labels(hdr, "Camera")
print(f"Loaded {len(camera_labels)} existing camera labels")

if camera_labels.empty:
    print("No project labels found. Add labels in the dashboard first, then rerun this cell.")


In [ ]:
add_project_labels(hdr, "Camera", labels=camera_labels)

## 2. Upload Timestamp Corrections

Use `update_media_timestamps(...)` with a list of `MediaTimestampUpdate` objects.

This example shifts a small sample of media timestamps by +1 minute.

In [ ]:
camera_obs = get_camera_trap_data(hdr)
if camera_obs.empty:
    print("No camera observations found.")
else:
    sample = camera_obs[["media_file_record_id", "media_file_created_at"]].dropna().drop_duplicates().head(5)
    updates = []
    for row in sample.itertuples(index=False):
        old_ts = pd.to_datetime(row.media_file_created_at, utc=True)
        new_ts = (old_ts + timedelta(minutes=1)).to_pydatetime()
        updates.append(MediaTimestampUpdate(media_file_record_id=int(row.media_file_record_id), new_timestamp=new_ts))

    print(f"Prepared {len(updates)} timestamp update record(s)")

    if RUN_UPLOADS and updates:
        results = update_media_timestamps(hdr, updates)
        print(f"API returned {len(results)} responses")
    else:
        print("Dry run only: set RUN_UPLOADS=True to call update_media_timestamps(...)")

## 3. Validate and Upload eDNA Records

Recommended flow:
1. Build eDNA rows in a DataFrame
2. Validate with `check_edna_labels_df(...)`
3. Upload only rows with `status == 'success'`

In [ ]:
edna_input = pd.DataFrame([
    {
        "marker_name": "COI",
        "sequence": "ATGCCGTAGCTA",
        "primer": "mlCOIintF",
        "timestamp": datetime.now(tz=timezone.utc),
        "genus": "Canis",
        "species": "Canis lupus",
        "confidence": 99,
    },
])

validated_df = check_edna_labels_df(hdr, edna_input)
display(validated_df)

successful_records = [
    eDNAUploadResponse.model_validate(record)
    for record in validated_df.to_dict(orient="records")
    if record.get("status") == "success"
]
print(f"Validated success rows: {len(successful_records)}")

edna_stations = get_station_info(hdr, measurement_type="edna")
target_psr = int(edna_stations.project_system_record_id.dropna().iloc[0]) if not edna_stations.empty else None
print(f"Target eDNA project_system_record_id: {target_psr}")

if RUN_UPLOADS and successful_records and target_psr is not None:
    upload_result = upload_edna_records(hdr, successful_records, project_system_record_id=target_psr)
    print(f"Uploaded/processed rows: {len(upload_result)}")
else:
    print("Dry run only: set RUN_UPLOADS=True to call upload_edna_records(...)")

## 4. Upload phone observations from CSV

This mirrors the R tutorial: fetch the project schema once, select a system/procedure, validate a CSV against that procedure, dry-run the payload, then upload to `uploadObservations`.

Every CSV needs a longitude, a latitude, and a timestamp column. Common spellings are matched automatically and case is ignored, so `lon`/`lng`, `lat`, and `timestamp`/`datetime` all work in place of `longitude`, `latitude`, and `recorded_at`.

Two layouts are supported and detected automatically:

**Wide** — one row per observation, with each form field as its own column. Column headers must match the procedure item names; any other columns are ignored and listed for you.

```
timestamp,latitude,longitude,label,MIN.N.ADULT
04/04/2025,50.9474,-0.7196,Vulpes vulpes,1
```

**Long** — one row per field value, using an `item_name` column (or `item_uuid`) to say which field the row is for, and `data` for the value. Add `numbers` for numeric values and `observation_id` to control grouping explicitly.

```
longitude,latitude,recorded_at,item_name,data
13.703612,0.931838,15/04/2026 08:39,Taxonomic label,Myrianthus
13.703612,0.931838,15/04/2026 08:39,Nom commun,Oboba
```

In long format, rows sharing coordinates and a timestamp (or the same `observation_id`) are grouped into one observation. Only simple point observations are supported today.


In [3]:
# Fetch systems/procedures for this project, then select the procedure that matches your CSV.
project_systems = get_project_systems(hdr)
systems_table = list_systems(project_systems)

 system_index                    system_name  system_id procedure_index           procedure_name procedure_id  form
            1   Song Meter Mini Bat 2 Li-ion        338            <NA>                      NaN         <NA>  <NA>
            2 Browning Recon Force Elite HP5        337               1     Environmental sample          157 False
            2 Browning Recon Force Elite HP5        337               2 Simple sensor deployment          210 False
            2 Browning Recon Force Elite HP5        337               3               Harsh Test          265 False
            2 Browning Recon Force Elite HP5        337               4            One Procedure          316 False
            2 Browning Recon Force Elite HP5        337               5           Procedure 1:21          366 False
            2 Browning Recon Force Elite HP5        337               6            Procedure One          416 False
            2 Browning Recon Force Elite HP5        337               7 

In [4]:
# Replace these with values from list_systems(...) for your project. You can select
# by name, by api id (system_id=437), or by 1-based position in the schema
# (system_index=1). The *_index arguments are positions, not ids.
SYSTEM_NAME = "Bird Survey"
PROCEDURE_NAME = "bird survey"

procedure = get_procedure(
    project_systems,
    system_id=437,
    procedure_id=866
)
print(f"Using system_id={procedure['system_id']}, procedure_id={procedure['procedure_id']}")
display(procedure["items"])

System: Bird Survey (id: 437)
Procedure: bird survey (id: 866, form: False)
Items (4):
 item_id                            item_uuid      item_name      item_description data_type  nullable                                                                                                 choices
    1751 4ee6e82b-9605-4a25-95c1-d41e13af5472          label          species name     label      True                                                                                                        
    1752 67685420-f061-4e15-96a3-dbd68df1467d       behavior                          choice      True singing | calling | alarm | family | pair | fly.over | land | fly.away | carry.nest | carry.food | nest
    1753 ec2aab9a-fa26-47cd-aae5-31297ec79144  n_individuals number of individuals   numeric      True                                                                                 male | female | unknown
    1754 e9f8d3ae-b90d-47bd-a675-1b9000f8bb22 sex_individual sex of the individual   

,item_id,item_uuid,item_name,item_description,data_type,nullable,choices
0,1751,4ee6e82b-9605-4a25-95c1-d41e13af5472,label,species name,label,True,
1,1752,67685420-f061-4e15-96a3-dbd68df1467d,behavior,,choice,True,singing | calling | alarm | family | pair | fl...
2,1753,ec2aab9a-fa26-47cd-aae5-31297ec79144,n_individuals,number of individuals,numeric,True,male | female | unknown
3,1754,e9f8d3ae-b90d-47bd-a675-1b9000f8bb22,sex_individual,sex of the individual,choice,True,male | female | unknown


### Validate the CSV against the procedure

Catch missing columns, unknown item names, and type issues before building the upload payload.

In [5]:
CSV_PATH = TUTORIAL_DIR / "data" / "bird_test.csv"

print(f"CSV path: {CSV_PATH}")
print(CSV_PATH.exists())

validation = validate_csv_against_procedure(
    procedure=procedure,
    csv_path=CSV_PATH,
    
)
print("valid:", validation["valid"])
if validation["issues"]:
    print("issues:")
    for issue in validation["issues"]:
        print("-", issue)


CSV path: /Users/natimi/Projects/NatureCubePy/tutorials/data/bird_test.csv
True
Detected CSV format: wide

--- CSV Validation Report (wide format) ---
System: Bird Survey (id: 437)
Procedure: bird survey (id: 866)
Matched items (4/4):
     item_name data_type
         label     label
      behavior    choice
 n_individuals   numeric
sex_individual    choice

Data type issues (657, showing first 15):
 row item_name data_type            value                                                                                                         problem
 100  behavior    choice     calling;land     multiple values 'calling;land' in a single choice cell are not supported; put each selection in its own row
 149  behavior    choice calling;fly.away multiple values 'calling;fly.away' in a single choice cell are not supported; put each selection in its own row
 151  behavior    choice calling;fly.away multiple values 'calling;fly.away' in a single choice cell are not supported; put each select

### Handle multi-select choice cells

If a `choice` column holds several selections in one cell (e.g. `calling;land`), validation flags it because the API stores one choice per item. Use `split_multi_value_choices(...)` to expand each selection into its own observation — the row's other fields (species label, count, coordinates, timestamp) are duplicated onto each, and each gets a unique `observation_id` so they upload as separate records.

Parts may be separated by `;` or `|`, casing is matched to the procedure's choices, and both wide and long layouts are supported.

In [6]:
observation_data = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

# Expand any multi-select choice cells (e.g. "calling;land") into one observation each.
expanded = split_multi_value_choices(observation_data, procedure=procedure)
print(f"Rows: {len(observation_data)} -> {len(expanded)} after splitting multi-select choices")

# Re-validate the expanded table (write to a temp file so the same checks run).
EXPANDED_CSV = TUTORIAL_DIR / "data" / "_expanded_observations.csv"
expanded.to_csv(EXPANDED_CSV, index=False)
validation = validate_csv_against_procedure(procedure=procedure, csv_path=EXPANDED_CSV)
print("valid after split:", validation["valid"])

Rows: 8221 -> 9029 after splitting multi-select choices
Detected CSV format: wide

--- CSV Validation Report (wide format) ---
System: Bird Survey (id: 437)
Procedure: bird survey (id: 866)
Matched items (4/4):
     item_name data_type
         label     label
      behavior    choice
 n_individuals   numeric
sex_individual    choice

Feature groups: 9029 parent feature(s)

Valid: True
Warnings (non-blocking):
  12 CSV column(s) not in procedure (ignored on upload): common_name, kingdom, phylum, class, order, family, genus, species, site_alias, site_id, observer, observation_id
valid after split: True


### Dry-run, then upload

Dry-run builds the `uploadObservations` payload without writing. Confirm `unresolved_rows` is empty before setting `RUN_UPLOADS = True`.

In [7]:
RUN_UPLOADS = False
# Must match how the timestamp column is written in your CSV, e.g.
# "%Y-%m-%dT%H:%M:%S" for 2015-04-24T07:15:00, "%d/%m/%Y" for 04/04/2025.
RECORDED_AT_FORMAT = "%Y-%m-%dT%H:%M:%S"

# Upload the expanded table so multi-select behaviors become separate observations.
dry_run_result = upload_observations_from_csv(
    hdr=hdr,
    csv_path=EXPANDED_CSV,
    procedure=procedure,
    dry_run=True,
    recorded_at_format=RECORDED_AT_FORMAT,
)

print(f"Format: {dry_run_result['format']}")
print(f"Built observations: {len(dry_run_result['observations'])}")
print(f"Resolved records: {dry_run_result['resolved_rows']} ({dry_run_result['resolved_values']} item values)")
print(f"Unresolved rows: {len(dry_run_result['unresolved_rows'])}")
if dry_run_result["ignored_columns"]:
    print(f"Columns not in the procedure (not uploaded): {dry_run_result['ignored_columns']}")
display(pd.DataFrame(dry_run_result["observations"]).head())

if RUN_UPLOADS and dry_run_result["unresolved_rows"].empty:
    upload_result = upload_observations_from_csv(
        hdr=hdr,
        csv_path=EXPANDED_CSV,
        procedure=procedure,
        dry_run=False,
        recorded_at_format=RECORDED_AT_FORMAT,
    )
    print(
        f"uploaded any: {upload_result['uploaded']} | "
        f"all succeeded: {upload_result['all_succeeded']} | "
        f"{upload_result['succeeded']} ok / {upload_result['failed']} failed"
    )
    if upload_result["errors"]:
        display(pd.DataFrame(upload_result["errors"]))
else:
    print("Dry run only. Set RUN_UPLOADS=True after unresolved_rows is empty to upload.")


Format: wide
Built observations: 9029
Resolved records: 9029 (35718 item values)
Unresolved rows: 0
Columns not in the procedure (not uploaded): ['common_name', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'site_alias', 'site_id', 'observer']


,survey_uuid,project_system_id,procedure_id,recorded_at,lon,lat,values
0,273538bd-58be-4f42-b682-46eec2a52d0d,437,866,2015-04-24T07:15:00Z,-1.432926,52.725012,{'4ee6e82b-9605-4a25-95c1-d41e13af5472': 'Sylv...
1,9c82d8a3-66b8-4b31-99c9-0d81cbb1cba2,437,866,2015-04-24T07:15:00Z,-1.433872,52.725554,{'4ee6e82b-9605-4a25-95c1-d41e13af5472': 'Sylv...
2,d7445bb3-66ef-4664-93a4-52b115edc557,437,866,2015-04-24T07:15:00Z,-1.436148,52.725415,{'4ee6e82b-9605-4a25-95c1-d41e13af5472': 'Sylv...
3,1cd8fa52-a2b6-498c-9b13-aca0eb3d4e87,437,866,2015-04-24T07:15:00Z,-1.436570,52.726747,{'4ee6e82b-9605-4a25-95c1-d41e13af5472': 'Sylv...
4,eb9d1ff4-8603-45b9-a6a3-6421a5249177,437,866,2015-04-24T07:15:00Z,-1.433437,52.726259,{'4ee6e82b-9605-4a25-95c1-d41e13af5472': 'Sylv...


Uploading batch 1/19 (500 observations, 500/9029 total)...
Uploading batch 2/19 (500 observations, 1000/9029 total)...
Uploading batch 3/19 (500 observations, 1500/9029 total)...
Uploading batch 4/19 (500 observations, 2000/9029 total)...
Uploading batch 5/19 (500 observations, 2500/9029 total)...
Uploading batch 6/19 (500 observations, 3000/9029 total)...
Uploading batch 7/19 (500 observations, 3500/9029 total)...
Uploading batch 8/19 (500 observations, 4000/9029 total)...
Uploading batch 9/19 (500 observations, 4500/9029 total)...
Uploading batch 10/19 (500 observations, 5000/9029 total)...
Uploading batch 11/19 (500 observations, 5500/9029 total)...
Uploading batch 12/19 (500 observations, 6000/9029 total)...
Uploading batch 13/19 (500 observations, 6500/9029 total)...
Uploading batch 14/19 (500 observations, 7000/9029 total)...
Uploading batch 15/19 (500 observations, 7500/9029 total)...
Uploading batch 16/19 (500 observations, 8000/9029 total)...
Uploading batch 17/19 (500 observa

### Optional: low-level phone feature upload

Use this path when you need explicit device settings and media files via `pushPhoneObservations`. Media is uploaded through signed GCS URLs first. For CSV point surveys, prefer `upload_observations_from_csv` above.

In [ ]:
PROJECT_ID = None  # numeric project id for pushPhoneObservations

device = build_device_settings(
    device_id=f"demo-{uuid.uuid4().hex[:8]}",
    phone_model="iPhone 14 Pro",
    phone_os="iOS 17",
    carrier="Demo Carrier",
    build_number="1.0.0",
    build_id="tutorial-build",
)

observation = build_observation(
    item_uuid=str(uuid.uuid4()),
    item_type="text",
    data=["Field note: observed signs of mammal activity near the stream."],
    geometry={"type": "Point", "coordinates": [-1.5, 53.4]},
)

feature = build_feature_record(
    feature_uuid=str(uuid.uuid4()),
    project_system_id=int(procedure["system_id"] or 1),
    procedure_id=int(procedure["procedure_id"] or 1),
    start_time=datetime.now(tz=timezone.utc) - timedelta(minutes=5),
    end_time=datetime.now(tz=timezone.utc),
    created_by_method="drawn",
    geometry={"type": "Point", "coordinates": [-1.5, 53.4]},
    observations=[observation],
)

validation = validate_observation_payload([feature], device)
print(validation)

if RUN_UPLOADS and validation["valid"] and PROJECT_ID is not None:
    upload_out = upload_phone_observations(
        hdr=hdr,
        project_id=int(PROJECT_ID),
        feature_payload=[feature],
        device_settings=device,
        validate=True,
    )
    print(upload_out["summary"])
else:
    print("Dry run only: set RUN_UPLOADS=True and PROJECT_ID to call upload_phone_observations(...)")


## Next Steps

- Point `SYSTEM_NAME` / `PROCEDURE_NAME` at your project schema.
- Replace the example CSV under `tutorials/data/` with your own long- or wide-format file.
- Re-run validation and dry-run until `unresolved_rows` is empty.
- Enable writes by setting `RUN_UPLOADS = True`.
